
# Άσκηση Αξιολόγησης 3

## Εκφώνηση

Θεωρήστε τις κυματοσυναρτήσεις:

$$
\psi(x,y)=\sin(2x)\cos(5x)
$$

$$
\phi(x,y)=e^{-2(x^2+y^2)}
$$

$$
\chi(x,y)=e^{-i(x+y)}
$$

και τους τελεστές:

$$
\hat{A}=\frac{\partial}{\partial x}+\frac{\partial}{\partial y}
$$

$$
\hat{B}=\frac{\partial^2}{\partial x^2}+\frac{\partial^2}{\partial y^2}+1
$$

### Ερωτήματα

**(α)** Εξετάστε αν κάποια από τις κυματοσυναρτήσεις είναι ιδιοσυνάρτηση του τελεστή $\hat{A}$.

**(β)** Βρείτε αν κάποια από τις κυματοσυναρτήσεις είναι ιδιοσυνάρτηση του τελεστή $\hat{B}$.

**(γ)** Υπολογίστε τις δράσεις των $\hat{A}\hat{B}$ και $\hat{B}\hat{A}$ σε κάθε μία από τις κυματοσυναρτήσεις και συναγάγετε το $[\hat{A},\hat{B}]$.

---

### Επιπλέον Βαθμός

**Όποιος λύσει την άσκηση χρησιμοποιώντας κλάση (class) στην Python, λαμβάνει επιπλέον +1 βαθμό.**


---
# Απάντηση #
---

Για να είναι μια κυματοσυνάρτηση ιδιοσυνάρτηση των τελετσών $\hat{A}$ ή/και $\hat{B}$ θα πρέπει να ικανοποιεί την εξίωσωση ιδιοτιμών, δηλαδή
$$\hat{A}\Psi(x,y)=\lambda\Psi(x,y)$$
όπου $\lambda$ είναι μια σταθερά

*Παρακάτω φαίνεται ολόκληρος ο κώδικας και εν συνεχεία παρουσιάζεται ξεχωριστά το κάθε του κομμάτι*

In [5]:
import sympy as sp 

x, y = sp.symbols('x y', real=True)

psi = sp.sin(2*x) * sp.cos(5*x)
phi = sp.exp(-2*(x**2 + y**2))
chi = sp.exp(-sp.I * (x + y))


class QuantumOperators:
    def __init__(self, name, action):
        self.name = name
        self.action = action
    
    def apply(self, wavefunction):
        return self.action(wavefunction)
    
    def eigenfunction(self, wavefunction):
        result = self.apply(wavefunction)
        value = sp.simplify(result / wavefunction)
        if not value.has(x,y):
            return True, value
        else:
            return False, None

def A_expr(f):
    return sp.diff(f, x) + sp.diff(f, y)

def B_expr(f):
    return sp.diff(f, x, 2) + sp.diff(f, y, 2) + f

operator_A = QuantumOperators("A", A_expr)
operator_B = QuantumOperators("B", B_expr)

wavefunctions = [("ψ", psi), ("φ", phi), ("χ", chi)]
operators = [operator_A, operator_B]

for op in operators:
    print(f"\n Έλεγχος του τελεστή {op.name}")

    for name, wf in wavefunctions:
        eigenfunction, val = op.eigenfunction(wf)
        if eigenfunction == True:
            print(f"Η {name} είναι ιδιοισυνάρτηση του {op.name}.")
        else:
            print(f"Η {name} δεν είναι ιδιοσυνάρτηση του {op.name}.")

class Commutator:
    def __init__(self, op_A, op_B):
        self.op_A = op_A
        self.op_B = op_B
        self.name = f"[{op_A.name}, {op_B.name}]"
    
    def apply(self, wavefunction):
        term_1 = self.op_A.apply(self.op_B.apply(wavefunction))
        term_2 = self.op_B.apply(self.op_A.apply(wavefunction))
        return sp.simplify(term_1 - term_2)

commutator_AB = Commutator(operator_A, operator_B)

wavefunctions = [("ψ", psi), ("φ", phi), ("χ", chi)]

for name, wf in wavefunctions:
    result = commutator_AB.apply(wf)
    print(f"Η δράση του {commutator_AB.name} στην {name} δίνει: {result}")


 Έλεγχος του τελεστή A
Η ψ δεν είναι ιδιοσυνάρτηση του A.
Η φ δεν είναι ιδιοσυνάρτηση του A.
Η χ είναι ιδιοισυνάρτηση του A.

 Έλεγχος του τελεστή B
Η ψ δεν είναι ιδιοσυνάρτηση του B.
Η φ δεν είναι ιδιοσυνάρτηση του B.
Η χ είναι ιδιοισυνάρτηση του B.
Η δράση του [A, B] στην ψ δίνει: 0
Η δράση του [A, B] στην φ δίνει: 0
Η δράση του [A, B] στην χ δίνει: 0


---------------------------------------------------------------------------------------------------
### Επεξήγηση ###
---------------------------------------------------------------------------------------------------

Εισαγωγή Βιβλιοθήκης και Ορισμός μεταβλητών και κυματοσυναρτήσεων

In [10]:
import sympy as sp 

x, y = sp.symbols('x y', real=True)

psi = sp.sin(2*x) * sp.cos(5*x)
phi = sp.exp(-2*(x**2 + y**2))
chi = sp.exp(-sp.I * (x + y))

Παρακάτω ορίζεται η κλάση `QuantumOperators` για την δράση των τελεστών στις κυματοσυναρτήσεις και τον έλεγχο - μέσω της `eigenfunction` - για το αν οι παραπάνω κυματοσυναρτήσεις αποτελούν ιδιοσυναρτήσεις των τελεστών Α και Β

In [11]:
class QuantumOperators:
    def __init__(self, name, action):
        self.name = name
        self.action = action
    
    def apply(self, wavefunction):
        return self.action(wavefunction)
    
    def eigenfunction(self, wavefunction):
        result = self.apply(wavefunction)
        value = sp.simplify(result / wavefunction)
        if not value.has(x,y):
            return True, value
        else:
            return False, None

def A_expr(f):
    return sp.diff(f, x) + sp.diff(f, y)

def B_expr(f):
    return sp.diff(f, x, 2) + sp.diff(f, y, 2) + f

operator_A = QuantumOperators("A", A_expr)
operator_B = QuantumOperators("B", B_expr)

## Ερωτήματα (α) και (β) ##
### Έλεγχος Ιδιοσυναρτήσεων ###
Παρακάτω ελέγχεται αν οι συναρτήσεις $\psi$, $\phi$ και $\chi$ είναι ιδιοσυναρτήσεις των $\hat{A}$ και $\hat{B}$

In [13]:
wavefunctions = [("ψ", psi), ("φ", phi), ("χ", chi)]
operators = [operator_A, operator_B]

for op in operators:
    print(f"\n Έλεγχος του τελεστή {op.name}")

    for name, wf in wavefunctions:
        eigenfunction, val = op.eigenfunction(wf)
        if eigenfunction == True:
            print(f"Η {name} είναι ιδιοσυνάρτηση του {op.name}.")
        else:
            print(f"Η {name} δεν είναι ιδιοσυνάρτηση του {op.name}.")


 Έλεγχος του τελεστή A
Η ψ δεν είναι ιδιοσυνάρτηση του A.
Η φ δεν είναι ιδιοσυνάρτηση του A.
Η χ είναι ιδιοσυνάρτηση του A.

 Έλεγχος του τελεστή B
Η ψ δεν είναι ιδιοσυνάρτηση του B.
Η φ δεν είναι ιδιοσυνάρτηση του B.
Η χ είναι ιδιοσυνάρτηση του B.


# Ερώτημα (γ) - Μεταθέτης #
Με την κλάση `Commutator` ελέγχεται για κάθε μια κυματοσυνάρτηση η τιμή του μεταθέτη $[\hat{A},\hat{B}]$

In [12]:
class Commutator:
    def __init__(self, op_A, op_B):
        self.op_A = op_A
        self.op_B = op_B
        self.name = f"[{op_A.name}, {op_B.name}]"
    
    def apply(self, wavefunction):
        term_1 = self.op_A.apply(self.op_B.apply(wavefunction))
        term_2 = self.op_B.apply(self.op_A.apply(wavefunction))
        return sp.simplify(term_1 - term_2)

commutator_AB = Commutator(operator_A, operator_B)

wavefunctions = [("ψ", psi), ("φ", phi), ("χ", chi)]

for name, wf in wavefunctions:
    result = commutator_AB.apply(wf)
    print(f"Η δράση του {commutator_AB.name} στην {name} δίνει: {result}")

Η δράση του [A, B] στην ψ δίνει: 0
Η δράση του [A, B] στην φ δίνει: 0
Η δράση του [A, B] στην χ δίνει: 0
